<a href="https://www.kaggle.com/code/anautiyal40/agrisense-ai?scriptVersionId=310793196" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install fastapi uvicorn nest-asyncio pyngrok python-multipart gradio pillow

In [3]:
from fastapi import FastAPI, File, UploadFile
import uvicorn
import nest_asyncio
from PIL import Image
import io

app = FastAPI()

@app.get("/")
def home():
    return {"message": "AgriSense AI API running"}

@app.post("/diagnose")
async def diagnose(file: UploadFile = File(...)):
    contents = await file.read()
    image = Image.open(io.BytesIO(contents))

    # Mock AI response (replace later with Gemma)
    return {
        "disease": "Leaf Blight",
        "confidence": "92%",
        "solution": "Use fungicide and remove infected leaves"
    }

@app.get("/ask")
def ask(q: str):
    if "rain" in q.lower():
        return {"answer": "Rain expected in 2 days. Avoid irrigation today."}
    return {"answer": "Conditions look good for your crops."}

nest_asyncio.apply()


In [4]:
import threading

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run).start()


In [5]:
import gradio as gr
import requests

API_URL = "http://localhost:8000"

def diagnose_image(image):
    with open(image, "rb") as f:
        res = requests.post(f"{API_URL}/diagnose", files={"file": f})
    data = res.json()

    return f"""
    🌿 Disease: {data['disease']}
    Confidence: {data['confidence']}
    
    Solution:
    {data['solution']}
    """

def ask_question(q):
    res = requests.get(f"{API_URL}/ask", params={"q": q})
    return res.json()["answer"]

with gr.Blocks() as demo:
    gr.Markdown("# 🌱 AgriSense AI")

    with gr.Tab("📸 Crop Scan"):
        img = gr.Image(type="filepath")
        output = gr.Textbox()
        btn = gr.Button("Diagnose")

        btn.click(diagnose_image, inputs=img, outputs=output)

    with gr.Tab("🎤 Ask Question"):
        q = gr.Textbox(label="Ask something...")
        ans = gr.Textbox()
        ask_btn = gr.Button("Ask")

        ask_btn.click(ask_question, inputs=q, outputs=ans)

demo.launch()

INFO:     Started server process [16]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


* Running on local URL:  http://127.0.0.1:7860
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://09b1a456e90a8f6081.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [6]:
!pip install fastapi uvicorn nest-asyncio pyngrok python-multipart gradio pillow gtts SpeechRecognition pydub

INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.3 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
  Attempting uninstall: typer
    Found existing installation: typer 0.24.1
    Uninstalling typer-0.24.1:
      Successfully uninstalled typer-0.24.1
  Attempting uninstall: typer-slim
    Found existing installation: typer-slim 0.24.0
    Uninstalling typer-slim-0.24.0:
      Successfully uninstalled typer-slim-0.24.0
ERROR

In [7]:
from fastapi import FastAPI, File, UploadFile
import uvicorn, nest_asyncio
from PIL import Image
import io

app = FastAPI()

@app.post("/diagnose")
async def diagnose(file: UploadFile = File(...)):
    contents = await file.read()
    image = Image.open(io.BytesIO(contents))

    return {
        "disease": "Leaf Blight",
        "confidence": "92%",
        "solution": "फफूंदनाशक का उपयोग करें और संक्रमित पत्तियों को हटाएं"
    }

@app.get("/ask")
def ask(q: str):
    if "बारिश" in q:
        return {"answer": "2 दिनों में बारिश होगी। आज सिंचाई न करें।"}
    return {"answer": "फसल की स्थिति अच्छी है। नियमित देखभाल जारी रखें।"}

nest_asyncio.apply()


In [8]:
import speech_recognition as sr
from gtts import gTTS
import os

# 🎤 Speech to Text (Hindi)
def speech_to_text(audio_file):
    r = sr.Recognizer()
    with sr.AudioFile(audio_file) as source:
        audio = r.record(source)
    try:
        text = r.recognize_google(audio, language="hi-IN")
        return text
    except:
        return "आवाज़ समझ नहीं आई"

# 🔊 Text to Speech (Hindi)
def text_to_speech(text):
    tts = gTTS(text=text, lang='hi')
    file_path = "output.mp3"
    tts.save(file_path)
    return file_path
    

In [9]:
import requests

API_KEY = "YOUR_API_KEY"

def get_weather(city="Delhi"):
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric"
    res = requests.get(url).json()

    temp = res["main"]["temp"]
    weather = res["weather"][0]["description"]

    return f"{city}: {temp}°C, {weather}"
    

In [10]:
import gradio as gr
import requests

API_URL = "http://localhost:8000"

def diagnose_image(image):
    with open(image, "rb") as f:
        res = requests.post(f"{API_URL}/diagnose", files={"file": f})
    data = res.json()

    return f"""
🌿 Disease: {data['disease']}
Confidence: {data['confidence']}

Solution:
{data['solution']}
"""

def ask_text(q):
    res = requests.get(f"{API_URL}/ask", params={"q": q})
    answer = res.json()["answer"]
    audio = text_to_speech(answer)
    return answer, audio

def ask_voice(audio_file):
    text = speech_to_text(audio_file)
    res = requests.get(f"{API_URL}/ask", params={"q": text})
    answer = res.json()["answer"]
    audio = text_to_speech(answer)
    return text, answer, audio

def weather_ui(city):
    return get_weather(city)

with gr.Blocks() as demo:
    gr.Markdown("# 🌱 AgriSense AI (Hindi Voice Enabled)")

    with gr.Tab("📸 Crop Scan"):
        img = gr.Image(type="filepath")
        output = gr.Textbox()
        btn = gr.Button("Diagnose")
        btn.click(diagnose_image, inputs=img, outputs=output)

    with gr.Tab("🎤 Ask (Text)"):
        q = gr.Textbox(label="प्रश्न पूछें")
        ans = gr.Textbox()
        audio = gr.Audio()
        btn = gr.Button("Ask")
        btn.click(ask_text, inputs=q, outputs=[ans, audio])

    with gr.Tab("🎤 Ask (Voice)"):
        audio_in = gr.Audio(type="filepath")
        text_out = gr.Textbox(label="आपने कहा")
        ans_out = gr.Textbox(label="उत्तर")
        audio_out = gr.Audio()
        btn2 = gr.Button("Speak")
        btn2.click(ask_voice, inputs=audio_in, outputs=[text_out, ans_out, audio_out])

    with gr.Tab("🌦️ Weather"):
        city = gr.Textbox(value="Delhi")
        weather = gr.Textbox()
        btn3 = gr.Button("Get Weather")
        btn3.click(weather_ui, inputs=city, outputs=weather)

demo.launch()


* Running on local URL:  http://127.0.0.1:7861
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://45c1cddab75059f039.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
